# Background LCI: unit burdens from ecoinvent and premise

Produces `data/background/`, the only place where Brightway, ecoinvent and premise are used:

| file | content |
|---|---|
| `unit_burdens.csv` | burden of one unit of every activity in `unit_burdens_dataSources` (materials, machine wear, electricity per country, waste treatment, avoided burdens) for all impact categories, for the baseline (ecoinvent as is) and every scenario of the `scenarios` tab (premise) |
| `materials_carbon.csv` | carbon content per kg wet material and rotation period (`benefits_constants_dataSources`; ecoinvent properties where the sheet says `to extract w/ bw`) |
| `unit_burden_samples_<scenario>.npz` | Monte Carlo samples of the unit burdens (ecoinvent uncertainty), for the scenarios flagged `in_results` |
| `external_<name>.csv` | burdens of activities that are not in ecoinvent (frozen; see `alternative_data_source` on the sheet) |

Everything that decides *what* is extracted comes from the sheet snapshot: the activity-to-dataset mapping, the scenarios (model, pathway, year), the ecoinvent version and system model, the impact method, the transport distance. **Needs**: an ecoinvent licence (`ECOINVENT_USERNAME`, `ECOINVENT_PASSWORD` in `.env`, only to import ecoinvent if it is not in the Brightway project yet) and a premise key (`PREMISE_KEY`, only to build a missing scenario database). The committed output of this notebook is enough to run everything else.

In [1]:
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
import bw2data as bd, bw2calc as bc, bw2io as bi
from dotenv import load_dotenv

from raw_lca import build_background as bb
from raw_lca.paths import BACKGROUND, newest_snapshot
from raw_lca.sheet import load_tables

load_dotenv(Path.cwd().parent / ".env")
PROJECT_NAME = "biobased_construction_impact_calculator"      # local Brightway project (machine-specific, not a model parameter)

tables = load_tables(newest_snapshot())
setup = dict(zip(tables["study_setup"]["parameter"], tables["study_setup"]["value"]))
const = dict(zip(tables["constants"]["constant"], tables["constants"]["value"]))
EI_VERSION, EI_MODEL, METHOD_FAMILY = str(setup["ecoinvent_version"]), str(setup["ecoinvent_system_model"]), str(setup["impact_method"])
EI_DB = f"ecoinvent-{EI_VERSION}-{EI_MODEL}"; BIOSPHERE = f"ecoinvent-{EI_VERSION}-biosphere"
TKM_PER_KG = float(const["transport_distance_km"]) * 0.001          # tonne-km per kg of collected material
scenarios = tables["scenarios"]
print(EI_DB, "|", METHOD_FAMILY, "| transport", const["transport_distance_km"], "km")

~/miniconda3/envs/brightway_env/lib/python3.11/site-packages/bw2calc/__init__.py:52: UserWarning: 
It seems like you have an AMD/INTEL x64 architecture, but haven't installed pypardiso:

    https://pypi.org/project/pypardiso/

Installing it could give you much faster calculations.

  warnings.warn(PYPARDISO_WARNING)


14:13:32+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


ecoinvent-3.12-cutoff | EF v3.1 | transport 50 km


## Brightway project and ecoinvent

In [2]:
bd.projects.set_current(PROJECT_NAME)
if EI_DB not in bd.databases:
    bi.import_ecoinvent_release(version=EI_VERSION, system_model=EI_MODEL,
                                username=os.environ["ECOINVENT_USERNAME"], password=os.environ["ECOINVENT_PASSWORD"])
METHODS = [m for m in bd.methods if m[0] == f"ecoinvent-{EI_VERSION}" and m[1] == METHOD_FAMILY]
UNITS = dict(zip(tables["impact_categories"]["category"], tables["impact_categories"]["unit"]))
missing_units = [m[2] for m in METHODS if m[2] not in UNITS]
assert not missing_units, f"impact_categories tab lacks {missing_units}"
print(f"{len(METHODS)} impact categories; ecoinvent datasets: {len(bd.Database(EI_DB))}")

25 impact categories; ecoinvent datasets: 26533


## Scenario databases (premise)

Built only if missing in the Brightway project. All sectors are updated.

In [3]:
prem = scenarios[scenarios["premise_model"].notna()]
to_build = [dict(model=r.premise_model, pathway=r.premise_pathway, year=int(r.premise_year)) for r in prem.itertuples()
            if bb.scenario_db_name(r.premise_model, r.premise_pathway, r.premise_year) not in bd.databases]
print(f"{len(prem) - len(to_build)} scenario database(s) exist, {len(to_build)} to build")
if to_build:
    from premise import NewDatabase
    ndb = NewDatabase(scenarios=to_build, source_db=EI_DB, source_version=EI_VERSION, source_type="brightway",
                      system_model=EI_MODEL, biosphere_name=BIOSPHERE, key=os.environ["PREMISE_KEY"])
    ndb.update()
    ndb.write_db_to_brightway(name=[bb.scenario_db_name(s["model"], s["pathway"], s["year"]) for s in to_build])

2 scenario database(s) exist, 0 to build


## Unit burdens

One row of `unit_burdens_dataSources` is one activity: an ecoinvent dataset (name, location, reference product), or an alternative source (road transport over `transport_distance_km`; a frozen external file). Datasets with a unit other than kg are converted with their `wet mass` property when the sheet asks for kg.

In [4]:
sources = bb.read_sources(tables)
europe = bb.european_locations(bd, EI_DB)
print(f"{len(sources)} activities on the sheet; {len(europe)} European countries for the electricity lookup")

frames = []
for r in scenarios.itertuples():
    db_name = EI_DB if pd.isna(r.premise_model) else bb.scenario_db_name(r.premise_model, r.premise_pathway, r.premise_year)
    frames.append(bb.extract_scenario(bd, bc, sources, r.scenario_id, db_name, EI_DB, METHODS, UNITS, TKM_PER_KG, BACKGROUND,
                                      f"{METHOD_FAMILY}", europe))
unit_burdens = pd.concat(frames, ignore_index=True)
unit_burdens = unit_burdens[["scenario", "material_name", "material_unit", "location", "impact_category", "impact_category_unit",
                             "score", "lca_database", "lca_method", "material_source"]]

# every scenario must contain the same activities
counts = unit_burdens.groupby("scenario")["material_name"].nunique()
assert counts.nunique() == 1, f"scenarios differ in the number of activities:\n{counts}"
unit_burdens.to_csv(BACKGROUND / "unit_burdens.csv", index=False)
print(counts.to_dict(), len(unit_burdens), "rows written")

48 activities on the sheet; 40 European countries for the electricity lookup


[baseline] 1/48 Peas (virgin)


[baseline] 2/48 Cotton fiber (virgin)


[baseline] 3/48 Hemp fiber (virgin)


[baseline] 4/48 Hard wood (virgin)


...

[image_SSP2-L_2050] 46/48 waste brick recycling (-)


[image_SSP2-L_2050] 47/48 waste plastic incineration (-)


[image_SSP2-L_2050] 48/48 waste plastic landfilling (-)


[image_SSP2-L_2050] 1 unresolved: ['CFW beam: no ecoinvent dataset and no alternative data source']


{'baseline': 42, 'image_SSP2-L_2050': 42, 'image_SSP2-M_2050': 42} 9375 rows written


## Carbon content and rotation period

In [5]:
carbon = bb.carbon_content(bd, tables["benefits_constants_dataSources"], EI_DB)
carbon.to_csv(BACKGROUND / "materials_carbon.csv", index=False)
carbon

,material_name,carbon_content_kgC_per_kgWet,rotation_period_yr
0,Peas,0.407697,1.0
1,Cotton fiber,0.438425,1.0
2,Hemp fiber,0.445704,1.0
3,Bark chips,0.205833,80.0
4,Sawdust,0.290588,80.0
5,Hard wood,0.290588,100.0
6,Soft wood,0.290588,65.0
7,Cellulose reject fibers,0.404000,NaN
8,Seagrass,0.336000,NaN
9,Fiber glue (epoxy resin),0.000000,NaN


## Background uncertainty (Monte Carlo)

For the scenarios flagged `in_results`, `background_mc_iterations` samples of every unit burden are drawn from ecoinvent's uncertainty data (technosphere and biosphere exchanges; characterisation factors are not sampled). Within one iteration all activities are solved on the same sampled matrices, so processes shared between materials are correlated. Electricity is sampled for the countries in `location_choices` only. The uncertainty analysis draws one iteration per parameter draw.

In [6]:
n_iter = int(setup["background_mc_iterations"]); seed = int(setup["mc_seed"])
mc_locations = [c for c in str(setup["location_choices"]).split(";") if c]
for r in scenarios[scenarios["in_results"].astype(str).str.upper() == "TRUE"].itertuples():
    db_name = EI_DB if pd.isna(r.premise_model) else bb.scenario_db_name(r.premise_model, r.premise_pathway, r.premise_year)
    samples = bb.sample_background(bd, bc, sources, db_name, EI_DB, METHODS, TKM_PER_KG, BACKGROUND, europe, mc_locations, n_iter, seed)
    samples["__categories__"] = np.array([m[2] for m in METHODS])
    np.savez_compressed(BACKGROUND / f"unit_burden_samples_{r.scenario_id}.npz", **samples)
    print(r.scenario_id, len(samples) - 1, "unit burdens x", n_iter, "iterations saved")

  ecoinvent-3.12-cutoff: 10/500 iterations


  ecoinvent-3.12-cutoff: 20/500 iterations


  ecoinvent-3.12-cutoff: 30/500 iterations


  ecoinvent-3.12-cutoff: 40/500 iterations


  ecoinvent-3.12-cutoff: 50/500 iterations


...

  premise-image-ssp2-l-2050: 470/500 iterations


  premise-image-ssp2-l-2050: 480/500 iterations


  premise-image-ssp2-l-2050: 490/500 iterations


  premise-image-ssp2-l-2050: 500/500 iterations


image_SSP2-L_2050 58 unit burdens x 500 iterations saved
